# CIA: Controllable Image Augmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fennecinspace/ciagen/blob/main/notebooks/CIA_Quickstart.ipynb)

[![arXiv](https://img.shields.io/badge/arXiv-2411.16128-blue)](https://arxiv.org/abs/2411.16128)

**CIA** (Controllable Image Augmentation) is a Python library for synthetic data augmentation using Stable Diffusion + ControlNet.

This notebook demonstrates how to:
1. Install CIA
2. Generate synthetic images from real images
3. Evaluate generation quality
4. Filter the best images

## 1. Installation

In [ ]:
!pip install ciagen

In [ ]:
import ciagen

print("Available functions:", ciagen.__all__)

## 2. Setup: Download sample images

We'll download a few sample images to use as source images.

In [ ]:
import urllib.request
from pathlib import Path

sample_dir = Path("sample_images")
sample_dir.mkdir(exist_ok=True)

sample_images = [
    ("https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/1.png", "1.png"),
    ("https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/2.png", "2.png"),
    ("https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/3.png", "3.png"),
]

for url, filename in sample_images:
    urllib.request.urlretrieve(url, sample_dir / filename)
    print(f"Downloaded {filename}")

print(f"\nDownloaded {len(list(sample_dir.glob('*')))} images to {sample_dir}")

## 3. Visualize source images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (img_path, _) in zip(axes, zip(sorted(sample_dir.glob("*")), sample_images)):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name)
    ax.axis("off")
plt.suptitle("Source Images", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Generate synthetic images

We use Canny edge detection as the control condition. This preserves the overall structure while allowing the model to generate new versions.

**Note**: This requires a GPU. If running on CPU, reduce `num_per_image` to 1.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cpu":
    print("Warning: GPU recommended for generation. This will be slow on CPU.")

In [ ]:
output_dir = Path("generated_images")
output_dir.mkdir(exist_ok=True)

result = ciagen.generate(
    source=str(sample_dir),
    output=str(output_dir),
    extractor="canny",
    sd_model="fennecinspace/sd-v15",
    cn_model="lllyasviel/sd-controlnet-canny",
    num_per_image=2,
    prompt="a high quality photo",
    seed=42,
    device=device,
)

print(f"\nGenerated {result['total_generated']} images")
print(f"Output: {result['output_path']}")

## 5. Visualize generated images

Compare the original and synthetic images side by side.

In [ ]:
gen_images = sorted(output_dir.glob("*.png"))
if gen_images:
    n_show = min(6, len(gen_images))
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, img_path in zip(axes.flat, gen_images[:n_show]):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=10)
        ax.axis("off")
    plt.suptitle("Generated Synthetic Images", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No generated images found.")

## 6. Evaluate quality

Compute FID (Fréchet Inception Distance) and Mahalanobis distance to assess how well the synthetic images match the real distribution.

In [ ]:
scores = ciagen.evaluate(
    real=str(sample_dir),
    generated=str(output_dir),
    metrics=["mld"],
    feature_extractor="vit",
)

print("Per-image Mahalanobis distances:")
for img_path, distance in scores['ptd']['mld'].items():
    print(f"  {Path(img_path).name}: {distance:.4f}")

## 7. Filter by quality

Keep only the best images based on the quality scores.

In [ ]:
kept = ciagen.filter_generated(
    generated=str(output_dir),
    method="top-k",
    value=3,
)

print("Filtered images:")
for metric_name, fe_data in kept.items():
    for fe_name, images in fe_data.items():
        print(f"  {metric_name}/{fe_name}: {len(images)} images kept")

## What's next?

- **[Documentation](https://ciagen.readthedocs.io)** — Full API reference and guides
- **[GitHub](https://github.com/fennecinspace/ciagen)** — Source code and examples
- **[arXiv Paper](https://arxiv.org/abs/2411.16128)** — Learn about the methodology

### Available extractors:

| Extractor | Description |
|-----------|-------------|
| `canny` | Canny edge detection (shown above) |
| `openpose` | Human pose estimation |
| `segmentation` | Semantic segmentation |
| `mediapipe_face` | Face landmarks |